<a href="https://colab.research.google.com/github/sairahul1526/pitch-deck-outline/blob/main/notebooks/ocr_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pitch-deck OCR benchmark

This notebook is a provider-neutral Colab runner for the private OCR sample. It keeps raw PDFs and gold annotations in your Google Drive and writes only hashed benchmark artifacts. Run the native baseline first; enable Docling or PaddleOCR only after the runtime is ready.

## 1. Runtime and data paths

Use a GPU runtime only for the optional VLM cells. The current local intake contains the public source snapshots; copy the private `data/raw` directory to a Drive folder before running this notebook. Do not upload permission emails or private gold transcriptions to GitHub.

In [ ]:
%pip install -q pypdf
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/pitch-deck-outline-data')
RAW_ROOT = DATA_ROOT / 'raw'
RUN_ROOT = DATA_ROOT / 'runs' / 'ocr'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('raw root:', RAW_ROOT)
print('exists:', RAW_ROOT.exists())

In [ ]:
awesome_pdfs = sorted((RAW_ROOT / 'awesome-pitch-decks' / 'pdfs').glob('*.pdf'))
pitch_deckz_root = RAW_ROOT / 'huggingface' / 'skyforclouds__pitch-deckz' / 'files'
pitch_deckz_pdfs = sorted(pitch_deckz_root.glob('*.pdf'))
print('Awesome Pitch Decks:', len(awesome_pdfs))
print('Pitch Deckz:', len(pitch_deckz_pdfs))
print('all PDFs:', len(awesome_pdfs) + len(pitch_deckz_pdfs))
assert awesome_pdfs, 'Copy the local raw data into the Drive path first'

## 2. Native PDF baseline

This is the speed baseline. It is intentionally conservative: pages with little extracted text should be routed to OCR rather than silently accepted.

In [ ]:
import hashlib
import json
import time

from pypdf import PdfReader


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def native_extract(path: Path) -> dict:
    started = time.perf_counter()
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        pages.append({'page_number': page_number, 'text': text, 'characters': len(text)})
    return {
        'engine': 'native-pypdf',
        'engine_version': 'colab-runtime',
        'source_file': str(path),
        'input_sha256': sha256_file(path),
        'latency_ms': round((time.perf_counter() - started) * 1000, 2),
        'pages': pages,
    }

sample = awesome_pdfs[:5]
native_results = [native_extract(path) for path in sample]
print([(Path(item['source_file']).name, len(item['pages'])) for item in native_results])

In [ ]:
native_report = RUN_ROOT / 'native-smoke.json'
native_report.write_text(json.dumps(native_results, indent=2) + '\n')
print(native_report)

## 3. Docling adapter (recommended first OCR candidate)

Docling is the first layout-aware candidate. It may install additional model assets on first use; keep its cache on the runtime or a disposable Drive cache.

In [ ]:
%pip install -q docling
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

def docling_extract(path: Path) -> dict:
    started = time.perf_counter()
    result = converter.convert(str(path))
    markdown = result.document.export_to_markdown()
    return {
        'engine': 'docling',
        'engine_version': 'colab-installed',
        'source_file': str(path),
        'input_sha256': sha256_file(path),
        'latency_ms': round((time.perf_counter() - started) * 1000, 2),
        'text': markdown,
    }

docling_result = docling_extract(sample[0])
print(docling_result['source_file'], len(docling_result['text']))

## 4. Optional PaddleOCR-VL adapter

Run this only on a GPU runtime after the native and Docling smoke checks pass. The API can change between PaddleOCR releases, so record the installed version in the report before comparing results.

In [ ]:
# Uncomment on a GPU runtime:
# %pip install -q paddlepaddle paddleocr
# from paddleocr import PaddleOCRVL
# vl_pipeline = PaddleOCRVL()
# vl_result = vl_pipeline.predict(str(sample[0]))
# print(vl_result)

## 5. Export a private benchmark artifact

Gold transcriptions and annotations are intentionally not generated here. Review the 50-page sample privately, then map each reviewed page into `BenchmarkCase` records in the repository's OCR harness.

In [ ]:
report = {
    'evidence_scope': 'colab_smoke',
    'native_results': native_results,
    'docling_result': docling_result,
    'notes': 'Smoke run only; not a quality claim.',
}
report_path = RUN_ROOT / 'ocr-smoke-report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n')
print(report_path)